# Hybrid Search with Qdrant

In [1]:
import os

import pandas as pd
from fastembed import LateInteractionTextEmbedding, SparseTextEmbedding, TextEmbedding
from qdrant_client import QdrantClient, models
from tqdm import tqdm

## Setup collection

**Note:** 
- The original parquet dataset is ~17MB, while the observed Qdrant collection is ~1.02GB
- There are 120,000 rows, each with an average of 236 characters

### Loading raw dataset

This code uses a function to download the dataset from Hugging Face and do some cleaning

In [2]:
# from scripts.get_dataset import get_dataset
# dataset = get_dataset()

This code uses the already downloaded dataset

In [3]:
dataset = pd.read_parquet(os.path.join(os.getcwd(), 'resources', 'dataset.parquet'))

print(f'Original dataset size: {dataset.shape[0]:,} rows')

dataset = (
    dataset[dataset['label'] == 'Sci/Tech']
    .sample(frac=0.1, random_state=42)
    .reset_index(drop=True)
)

print(f'Filtered dataset size: {dataset.shape[0]:,} rows')

dataset.head(10)

Original dataset size: 120,000 rows
Filtered dataset size: 3,000 rows


,text,label
0,Gartner optimistic about chip numbers But that...,Sci/Tech
1,Fossil Indicates Brown Bears Went South CALGAR...,Sci/Tech
2,Cryptic responds! As reported by The Associate...,Sci/Tech
3,Mozilla backtracks in eBay privacy flap The Mo...,Sci/Tech
4,New technique to enhance semiconductors Conven...,Sci/Tech
5,Newfound Comet Set for Winter Display (SPACE.c...,Sci/Tech
6,Intel and AMD battle for dual-core chips Advan...,Sci/Tech
7,Vodafone Peddles New BlackBerry Research In Mo...,Sci/Tech
8,US rocket team halfway to winning \$10-million...,Sci/Tech
9,The IPod's New Trick: Photo Show The iPod Phot...,Sci/Tech


In [4]:
# Text lenght analysis
dataset['text'].apply(len).describe()

count    3000.000000
mean      235.263000
std        78.503822
min       100.000000
25%       188.000000
50%       230.000000
75%       267.000000
max       996.000000
Name: text, dtype: float64

### Upserting data

In [5]:
qdrant_client = QdrantClient(url='http://localhost:6333')

Sets the embedding models to be used for dense, sparse, and late interaction embeddings.

In [6]:
DENSE_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
SPARSE_MODEL_NAME = 'Qdrant/bm25'
LATE_INTERACTION_MODEL_NAME = 'colbert-ir/colbertv2.0'

dense_embedding_model = TextEmbedding(DENSE_MODEL_NAME)
sparse_embedding_model = SparseTextEmbedding(SPARSE_MODEL_NAME)
late_interaction_embedding_model = LateInteractionTextEmbedding(
    LATE_INTERACTION_MODEL_NAME
)

#### Converts the dataset texts into vectors

In [7]:
documents = dataset['text'].values

dense_embeddings = list(
    dense_embedding_model.embed(
        tqdm(
            (doc for doc in documents),
            desc='Embedding documents with dense model',
            total=dataset.shape[0],
        ),
    )
)

sparse_embeddings = list(
    sparse_embedding_model.embed(
        tqdm(
            (doc for doc in documents),
            desc='Embedding documents with sparse model',
            total=dataset.shape[0],
        ),
    )
)

late_interaction_embeddings = list(
    late_interaction_embedding_model.embed(
        tqdm(
            (doc for doc in documents),
            desc='Embedding documents with late interaction model',
            total=dataset.shape[0],
        ),
    )
)

Embedding documents with late interaction model: 100%|██████████| 3000/3000 [04:46<00:00, 10.47it/s]


#### Creates the collection

In [8]:
COLLECTION_NAME = 'news_hybrid_search'

qdrant_client.create_collection(
    COLLECTION_NAME,
    # Configures each model with its vector size and distance metric
    # All models have independent configurations
    vectors_config={
        DENSE_MODEL_NAME: models.VectorParams(
            size=len(dense_embeddings[0]),
            distance=models.Distance.COSINE,
        ),
        LATE_INTERACTION_MODEL_NAME: models.VectorParams(
            size=len(late_interaction_embeddings[0][0]),
            distance=models.Distance.COSINE,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM,
            ),
        ),
    },
    sparse_vectors_config={
        SPARSE_MODEL_NAME: models.SparseVectorParams(modifier=models.Modifier.IDF)
    },
)

True

#### Do the upsert

In [9]:
points = []
for idx, (dense_embedding, sparse_embedding, late_interaction_embedding, label) in enumerate(
        zip(
            dense_embeddings,
            sparse_embeddings,
            late_interaction_embeddings,
            dataset['label'].values,
        )
    ):
    point = models.PointStruct(
        # Unique identifier for each vector
        id=idx,

        # Embedding vectors for the point
        vector={
            DENSE_MODEL_NAME: dense_embedding,
            SPARSE_MODEL_NAME: sparse_embedding.as_object(),
            LATE_INTERACTION_MODEL_NAME: late_interaction_embedding,
        },

        # Metadata for the point
        payload={'news_label': label},
    )
    points.append(point)

In [10]:
# Batching is recommended for large datasets to avoid the
# payload size limit of Qdrant (32MB)
BATCH_SIZE = 16

for i in tqdm(range(0, len(points), BATCH_SIZE), desc="Upserting in batches"):
    batch = points[i:i + BATCH_SIZE]
    qdrant_client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch,
        wait=True
    )

Upserting in batches: 100%|██████████| 188/188 [00:46<00:00,  4.02it/s]
